# Exploração de dados — Mercado de biometano no Brasil (ago/2026)

**Projeto Integrador V-A · PUC-Goiás · Big Data e Inteligência Artificial**  
Organização parceira: 4WaTT Bio Engenharia S/A

Este notebook documenta a análise exploratória (EDA) das bases organizadas em `dados/limpos/`, oriundas do **Boletim Mensal de Acompanhamento da Indústria de Biometano no Brasil — Ago/2026 (Ed. Nº 2)**, IEPUC-PUC-Rio, com dados da ANP, MME e B3.

In [1]:
from pathlib import Path
import pandas as pd

# __file__ não existe no Jupyter: sobe a partir do diretório atual até achar dados/limpos
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "limpos").is_dir())
D = BASE / "dados" / "limpos"

pm  = pd.read_csv(D / "producao_mensal_2026.csv")          # produção mensal 2026 × 2025
pe  = pd.read_csv(D / "producao_por_estado_2026_08.csv")   # desagregação por UF (ago/26)
mp  = pd.read_csv(D / "producao_por_materia_prima_2026_08.csv")
pu  = pd.read_csv(D / "producao_por_usina_2026_08.csv")    # por usina (ago/26)
cap = pd.read_csv(D / "capacidade_anp.csv")                # capacidade autorizada/tramitação

pm.head()

,mes,periodo,producao_mm3d,variacao_vs_mes_anterior_pct,media_acumulada_2026_mm3d,crescimento_vs_ano_anterior_pct
0,Jan,2026-01,384,-0.57,384,57.68
1,Fev,2026-02,410,6.90,396,67.06
2,Mar,2026-03,404,-1.54,399,62.11
3,Abr,2026-04,451,11.64,412,61.27
4,Mai,2026-05,451,0.12,420,56.97


## Perguntas de análise (KPIs do projeto)

1. Qual a tendência da produção nacional em 2026 e em que ritmo o mercado cresce?
2. Como a produção se distribui por estado, matéria-prima e usina (concentração)?
3. Qual o pipeline de capacidade futura (autorizada × em tramitação na ANP)?
4. O que isso significa para a 4WaTT (regiões e substratos com espaço de crescimento)?

In [2]:
import plotly.graph_objects as go

# 2025 = série real dos dados abertos da ANP (soma das UFs). A série "derivada" do crescimento
# a.a. publicado pelo IEPUC diverge até -11% (ago/25): a base de 2025 difere entre as fontes.
hist = pd.read_csv(D / "producao_nacional_anp_historico.csv")
anp = dict(zip(hist["periodo"], hist["producao_mm3d"]))
pm["producao_2025_anp"] = pm["periodo"].map(lambda p: anp[f"2025-{p[5:]}"])
pm["producao_2025_derivada"] = pm["producao_mm3d"] / (1 + pm["crescimento_vs_ano_anterior_pct"] / 100)
print((pm[["mes", "producao_2025_anp", "producao_2025_derivada"]].round(1)).to_string(index=False))
print(f"\nMédia jan-ago/26 (publicada): 466 mil m³/d | 2025 (ANP): {pm.producao_2025_anp.mean():.0f} "
      f"→ {(466 / pm.producao_2025_anp.mean() - 1) * 100:+.1f}%")
print(f"Ago/26 vs ago/25: {(529 / anp['2025-08'] - 1) * 100:+.1f}% (ANP) | +53,74% (publicado pelo IEPUC)")
print(f"Desde jan/2020: {anp['2020-01']} → {anp['2026-08']} mil m³/d ({anp['2026-08'] / anp['2020-01']:.1f}×)")

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist["periodo"], y=hist["producao_mm3d"], mode="lines", name="ANP (2020–2026)"))
fig.update_layout(height=380, title="Produção nacional de biometano — dados abertos ANP (mil m³/d)",
                  yaxis_title="mil m³/d")
fig.show()


mes  producao_2025_anp  producao_2025_derivada
Jan              243.3                   243.5
Fev              230.4                   245.4
Mar              262.9                   249.2
Abr              283.4                   279.7
Mai              314.7                   287.3
Jun              327.5                   336.7
Jul              367.8                   357.3
Ago              386.1                   344.1

Média jan-ago/26 (publicada): 466 mil m³/d | 2025 (ANP): 302 → +54.3%
Ago/26 vs ago/25: +37.0% (ANP) | +53,74% (publicado pelo IEPUC)
Desde jan/2020: 69.7 → 529.1 mil m³/d (7.6×)


In [3]:
# Concentração por estado
pe_top = pe[pe["uf"] != "OUTROS"].sort_values("producao_mm3d", ascending=False)
top5 = pe_top.head(5)["producao_mm3d"].sum()
total = pe["producao_mm3d"].sum()
print(f"Top 5 estados: {top5:.0f} Mm³/d = {top5/total*100:.1f}% da produção nacional")
print(pe_top)

# Concentração por matéria-prima
print("\nPor matéria-prima:")
print(mp.set_index("materia_prima")["participacao_pct"])

Top 5 estados: 524 Mm³/d = 99.1% da produção nacional
   uf             estado  num_usinas  producao_mm3d  var_mensal_pct  \
1  SP          São Paulo           9            178            -4.2   
0  RJ     Rio de Janeiro           2            139             2.4   
3  RS  Rio Grande do Sul           3             86             2.6   
4  PE         Pernambuco           1             84             8.4   
2  CE              Ceará           1             37           -37.9   

   media_no_ano_mm3d  cresc_vs_ano_anterior_pct  
1                138                       31.0  
0                128                        0.0  
3                 71                        NaN  
4                 50                        NaN  
2                 65                        8.0  

Por matéria-prima:
materia_prima
Aterro sanitário    80.3
Sucroenergético     18.8
Outros resíduos      1.0
Name: participacao_pct, dtype: float64


In [4]:
# Pipeline de capacidade (ANP)
cap

,periodo,autorizada_mm3d,solicitada_mm3d,total_mm3d,usinas_concedidas,pedidos_em_tramitacao
0,2025-06,697,1510,2207,12,38
1,2026-07,1368,2009,3378,21,49
2,2026-08,1368,2023,3391,21,49


## Utilização da capacidade e o caso de Goiás

Duas bases complementares: a taxa de utilização da capacidade autorizada (boletim IEPUC) e o potencial teórico de biogás de Goiás, estado-sede da 4WaTT (*Panorama do Biometano em Goiás*, Governo de Goiás/CBIE, 2026).

In [5]:
ut = pd.read_csv(D / "utilizacao_capacidade_2026_08.csv")
go_fonte = pd.read_csv(D / "potencial_biogas_goias.csv")
go_mun = pd.read_csv(D / "potencial_biogas_goias_municipios.csv")

print(ut[["recorte", "taxa_utilizacao_pct"]].to_string(index=False))

tot = go_fonte["potencial_biogas_m3_ano"].sum()
go_fonte["participacao_pct"] = (go_fonte["potencial_biogas_m3_ano"] / tot * 100).round(1)
print(f"\nPotencial teórico de biogás em Goiás: {tot/1e9:.2f} bi m³/ano")
print(go_fonte[["fonte", "potencial_biogas_m3_ano", "participacao_pct"]].to_string(index=False))

# escala: 1,7 bi m³/ano de biometano (estimativa do estudo) em mil m³/d vs produção nacional
bio_go = 1.7e9 / 365 / 1000
print(f"\nBiometano equivalente: {bio_go:,.0f} mil m³/d ≈ {bio_go / 529:.1f}× a produção nacional de ago/26")
go_mun.sort_values("rsu_m3_ano", ascending=False).head(3)[["municipio", "rsu_m3_ano"]]


                                 recorte  taxa_utilizacao_pct
                          Brasil (total)                 38.7
Plantas com 1 ano ou mais de autorização                 51.5
                        Aterro sanitário                 47.4
                         Sucroenergético                 27.2
                         Outros resíduos                  4.9

Potencial teórico de biogás em Goiás: 2.73 bi m³/ano
                                                     fonte  potencial_biogas_m3_ano  participacao_pct
Sucroenergético (vinhaça; torta de filtro; palha e bagaço)               1959230944              71.9
                 Pecuária (aves; suínos; vacas de ordenha)                401991067              14.8
                                  Resíduos sólidos urbanos                364016394              13.4

Biometano equivalente: 4,658 mil m³/d ≈ 8.8× a produção nacional de ago/26


,municipio,rsu_m3_ano
0,Goiânia,124128963
4,Itumbiara,6934200
12,Rio Verde,6854400


## Principais achados

- **Mercado em expansão acelerada**: média jan–ago/26 (466 mil m³/d) ~54% acima do mesmo período de 2025 (dados abertos ANP; o IEPUC publica +53,7% a.a. em ago/26, sobre outra base de 2025) e 7,6× a produção de jan/2020; capacidade autorizada praticamente dobrou em um ano (697 → 1.368 Mm³/d, +96%) e 49 pedidos somam mais 2.023 Mm³/d (projeção IEPUC: 3.391 Mm³/d até dez/2028).
- **Concentração geográfica altíssima**: SP (33,6%) + RJ (26,3%) ≈ 60%; top 5 estados ≈ 99% da produção. Goiás — sede da 4WaTT — não produz biometano.
- **Concentração de matéria-prima**: aterros sanitários = 80,3%; substratos agrícolas/industriais (~20%) são sub-representados; a fatia não sucroenergética ("Outros resíduos") é ~1% e instável (-91% em ago/26).
- **Capacidade ociosa**: o parque de usinas opera a 38,7% da capacidade autorizada (sucroenergético 27,2%) — espaço para O&M.
- **Goiás**: potencial teórico de 2,7 bi m³/ano de biogás (72% sucroenergético), sem produção de biometano; Goiânia lidera o potencial de RSU (34% do estado).
- **Estrutura competitiva**: Gás Verde é o líder isolado (139,48 Mm³/d ≈ 26% da produção nacional), seguido de Orizon Jaboatão (~16%) e Raízen-Geo (~14%).

> Todos os números foram validados programaticamente contra o boletim: `notebook/validar_dados.py` (83 verificações, 0 falhas — inclusive busca literal dos valores no texto dos PDFs).

### Implicação para a 4WaTT
O cruzamento (regiões fora do grupo de líderes × substratos agrícolas/industriais) delimita o espaço de crescimento natural para uma empresa com engenharia própria de EPC + O&M e operação já instalada em GO.


## Fontes e limitações

- IEPUC-PUC-Rio, *Boletim Mensal de Acompanhamento da Indústria de Biometano no Brasil*, Ago/2026, Ed. Nº 2 (dados ANP/MME/B3).
- Unidade do boletim: **Mm³/d = mil m³ por dia** (notação ANP/MME).
- Base de transações do IEPUC não é exaustiva; registro ANP cobre apenas contratos dentro da especificação regulatória.
- Dados da 4WaTT exclusivamente de fontes públicas (site oficial).